# MBG Reply Pipeline — Colab
**Process 509k reply tweets → sentiment-analyzed CSV**

### Before you start:
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Fill in credentials in **Cell 3**
3. Run all cells top-to-bottom

### Pipeline stages:
| Stage | Script | Output |
|---|---|---|
| R1 | `r1_jsonl_to_csv.py` | `replies_raw.csv` |
| R2 | `r2_enrich_metadata.py` | `replies_enriched.csv` |
| R3 | `r3_add_depth.py` | `replies_depth.csv` |
| R4 | `r4_filter_text.py` | `replies_filtered.csv` |
| R5 | `r5_tag_language.py` | `replies_tagged.csv` |
| R6 | `r6_preprocess_text.py` | `replies_preprocessed.csv` |
| R7 | `r7_sentiment.py` | `replies_sentiment.csv` ✓ |

**Note**: No relevance filtering — replies inherit context from parents

## Cell 1 — Mount Drive & Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time

notebook_start_time = time.time()
os.environ["RUNTIME_MODE"] = "colab"

DRIVE_BASE = "/content/drive/MyDrive/mbg"
DATA_DIR = f"{DRIVE_BASE}/data"
REPLY_DIR = f"{DATA_DIR}/replies"
OUTPUT_DIR = f"{DATA_DIR}/output"
CODE_DIR = "/content/mbg-pipeline"

for d in [REPLY_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Drive mounted")
print(f"   Reply data: {REPLY_DIR}")
print(f"   Outputs:    {OUTPUT_DIR}")
print(f"   Code:       {CODE_DIR}")

## Cell 2 — Verify GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "❌ No GPU — go to Runtime → Change runtime type → T4 GPU"
print(f"✅ GPU ready: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Cell 3 — Clone Codebase

In [ ]:
GITHUB_REPO = "https://github.com/FatwaArya/mbg-analysis"

import subprocess, sys

if not os.path.exists(CODE_DIR):
    print("Cloning repo...")
    subprocess.run(["git", "clone", GITHUB_REPO, CODE_DIR], check=True)
else:
    print("Repo exists — pulling latest...")
    subprocess.run(["git", "-C", CODE_DIR, "pull"], check=True)

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

print(f"✅ Codebase ready at {CODE_DIR}")

## Cell 4 — Install Dependencies

In [ ]:
import subprocess

req_path = f"{CODE_DIR}/requirements.txt"
print("Installing dependencies (~3-5 min)...")
subprocess.run(["pip", "install", "-q", "-r", req_path], check=True)
print("✅ Dependencies installed")

## Cell 5 — Download Reply Data from DO Spaces

In [ ]:
REPLY_JSONL = f"{REPLY_DIR}/replies_all_dedup.jsonl"

if os.path.exists(REPLY_JSONL):
    import os
    size_mb = os.path.getsize(REPLY_JSONL) / 1e6
    print(f"✅ Reply data found: {size_mb:.0f}MB")
else:
    print("Downloading reply data from DO Spaces (~276MB, ~2-3 min)...")
    !pip install -q s3cmd
    
    from google.colab import userdata
    s3cfg = f'[default]\naccess_key = {userdata.get("DO_ACCESS_KEY")}\nsecret_key = {userdata.get("DO_SECRET_KEY")}\nhost_base = sgp1.digitaloceanspaces.com\nhost_bucket = %(bucket)s.sgp1.digitaloceanspaces.com\n'
    open('/root/.s3cfg', 'w').write(s3cfg)
    
    !s3cmd get s3://mbg-scraper-network-20260419071440/replies_all_dedup.jsonl {REPLY_JSONL}
    print(f"✅ Downloaded to {REPLY_JSONL}")

## Cell 6 — Download Parent Posts (for metadata enrichment)

In [ ]:
PARENTS_CSV = f"{OUTPUT_DIR}/tweets_relevant.csv"

if os.path.exists(PARENTS_CSV):
    import pandas as pd
    df = pd.read_csv(PARENTS_CSV)
    print(f"✅ Parent posts found: {len(df):,} rows")
else:
    print("❌ Parent posts not found")
    print(f"   Upload tweets_relevant.csv to {OUTPUT_DIR}")
    print("   Or run parent pipeline first")
    raise FileNotFoundError("Parent posts required for R2")

## R1 — JSONL to CSV

In [ ]:
R1_OUT = f"{REPLY_DIR}/replies_raw.csv"

if os.path.exists(R1_OUT):
    import pandas as pd
    df = pd.read_csv(R1_OUT)
    print(f"⏭️  Skipping R1 — output exists ({len(df):,} rows)")
else:
    print("Running R1: JSONL → CSV...")
    t0 = time.time()
    !python3 {CODE_DIR}/scripts/replies/r1_jsonl_to_csv.py {REPLY_JSONL}
    elapsed = time.time() - t0
    
    df = pd.read_csv(R1_OUT)
    print(f"✅ R1 done in {elapsed:.0f}s — {len(df):,} rows")

## R2 — Metadata Enrichment

In [ ]:
R2_OUT = f"{REPLY_DIR}/replies_enriched.csv"

if os.path.exists(R2_OUT):
    import pandas as pd
    df = pd.read_csv(R2_OUT)
    print(f"⏭️  Skipping R2 — output exists ({len(df):,} rows)")
else:
    print("Running R2: Metadata enrichment...")
    t0 = time.time()
    
    # Modify script to use Colab paths
    import pandas as pd
    replies = pd.read_csv(R1_OUT, dtype={"id": str, "parent_id": str})
    parents = pd.read_csv(PARENTS_CSV, dtype={"id": str}, usecols=["id", "created_at", "lang", "date", "hour", "query_raw", "scrape_tab"])
    
    parent_meta = parents.rename(columns={"id": "parent_id", "created_at": "parent_created_at", "lang": "parent_lang", "date": "parent_date", "hour": "parent_hour", "query_raw": "parent_query_raw", "scrape_tab": "parent_scrape_tab"})
    replies = replies.merge(parent_meta, on="parent_id", how="left")
    replies["created_at"] = replies["created_at"].fillna(replies["parent_created_at"])
    replies["lang"] = replies["lang"].fillna(replies["parent_lang"])
    replies["created_at"] = pd.to_datetime(replies["created_at"], errors="coerce")
    replies["date"] = replies["created_at"].dt.date.astype(str)
    replies["hour"] = replies["created_at"].dt.hour
    replies["date"] = replies["date"].fillna(replies["parent_date"])
    replies["hour"] = replies["hour"].fillna(replies["parent_hour"])
    replies = replies.drop(columns=["parent_created_at", "parent_lang", "parent_date", "parent_hour", "parent_query_raw", "parent_scrape_tab"], errors="ignore")
    replies.to_csv(R2_OUT, index=False)
    
    elapsed = time.time() - t0
    print(f"✅ R2 done in {elapsed:.0f}s — {len(replies):,} rows")
    print(f"   Nulls: created_at={replies['created_at'].isna().sum()}, lang={replies['lang'].isna().sum()}")

## R3 — Depth Classification

In [ ]:
R3_OUT = f"{REPLY_DIR}/replies_depth.csv"

if os.path.exists(R3_OUT):
    import pandas as pd
    df = pd.read_csv(R3_OUT)
    print(f"⏭️  Skipping R3 — output exists ({len(df):,} rows)")
else:
    print("Running R3: Depth classification...")
    t0 = time.time()
    
    import pandas as pd
    replies = pd.read_csv(R2_OUT, dtype={"id": str, "parent_id": str})
    parents = pd.read_csv(PARENTS_CSV, dtype={"id": str}, usecols=["id"])
    
    parent_ids = set(parents["id"].astype(str))
    reply_ids = set(replies["id"].astype(str))
    
    def classify_depth(pid):
        pid = str(pid)
        if pid in parent_ids:
            return 1
        elif pid in reply_ids:
            return 2
        return 0
    
    replies["depth"] = replies["parent_id"].apply(classify_depth)
    replies.to_csv(R3_OUT, index=False)
    
    elapsed = time.time() - t0
    print(f"✅ R3 done in {elapsed:.0f}s — {len(replies):,} rows")
    print(replies["depth"].value_counts().to_string())

## R4 — Text Filtering

In [ ]:
R4_OUT = f"{REPLY_DIR}/replies_filtered.csv"

if os.path.exists(R4_OUT):
    import pandas as pd
    df = pd.read_csv(R4_OUT)
    print(f"⏭️  Skipping R4 — output exists ({len(df):,} rows)")
else:
    print("Running R4: Text filtering...")
    t0 = time.time()
    !python3 {CODE_DIR}/scripts/replies/r4_filter_text.py {R3_OUT}
    elapsed = time.time() - t0
    
    df = pd.read_csv(R4_OUT)
    print(f"✅ R4 done in {elapsed:.0f}s — {len(df):,} rows")

## R5 — Language Detection

In [ ]:
R5_OUT = f"{REPLY_DIR}/replies_tagged.csv"

if os.path.exists(R5_OUT):
    import pandas as pd
    df = pd.read_csv(R5_OUT)
    print(f"⏭️  Skipping R5 — output exists ({len(df):,} rows)")
else:
    print("Running R5: Language detection (~30-60s)...")
    t0 = time.time()
    !python3 {CODE_DIR}/scripts/replies/r5_tag_language.py {R4_OUT}
    elapsed = time.time() - t0
    
    df = pd.read_csv(R5_OUT)
    print(f"✅ R5 done in {elapsed:.0f}s — {len(df):,} rows")
    print(df["detected_lang"].value_counts().head().to_string())

## R6 — Text Preprocessing

In [ ]:
R6_OUT = f"{REPLY_DIR}/replies_preprocessed.csv"

if os.path.exists(R6_OUT):
    import pandas as pd
    df = pd.read_csv(R6_OUT)
    print(f"⏭️  Skipping R6 — output exists ({len(df):,} rows)")
else:
    print("Running R6: Text preprocessing (~2-5 min)...")
    t0 = time.time()
    !python3 {CODE_DIR}/scripts/replies/r6_preprocess_text.py {R5_OUT}
    elapsed = time.time() - t0
    
    df = pd.read_csv(R6_OUT)
    print(f"✅ R6 done in {elapsed/60:.1f} min — {len(df):,} rows")

## R7 — Sentiment Analysis (GPU, ~30-60 min)

In [ ]:
R7_OUT = f"{REPLY_DIR}/replies_sentiment.csv"

if os.path.exists(R7_OUT):
    import pandas as pd
    df = pd.read_csv(R7_OUT)
    print(f"⏭️  Skipping R7 — output exists ({len(df):,} rows)")
else:
    print("Running R7: Sentiment analysis (GPU, ~30-60 min for 450k replies)...")
    t0 = time.time()
    !python3 {CODE_DIR}/scripts/replies/r7_sentiment.py {R6_OUT}
    elapsed = time.time() - t0
    
    df = pd.read_csv(R7_OUT)
    print(f"\n✅ R7 done in {elapsed/60:.1f} min — {len(df):,} rows")
    print("\nSentiment distribution:")
    print(df["sentiment_normalized"].value_counts().to_string())

## Pipeline Complete — Summary

In [ ]:
import pandas as pd

total_time = (time.time() - notebook_start_time) / 60

print("=" * 50)
print("MBG REPLY PIPELINE COMPLETE")
print("=" * 50)
print(f"Total runtime: {total_time:.1f} minutes")
print(f"\nFinal output: {R7_OUT}")

df_final = pd.read_csv(R7_OUT)
print(f"\nRows: {len(df_final):,}")
print(f"Columns: {len(df_final.columns)}")
print(f"\nSentiment breakdown:")
print(df_final["sentiment_normalized"].value_counts().to_string())
print(f"\nDepth breakdown:")
print(df_final["depth"].value_counts().to_string())

print(f"\n✅ Ready for analysis/dashboard integration")